# Plan-Based Orchestration: Creating a Mini-Lesson

This notebook demonstrates a plan-based autonomous orchestration pattern.

Scenario:

A planner coordinates several specialist agents to create a mini-lesson.

Specialists:

- Learning Designer
- Math Explanation Agent
- Python Coding Agent
- Assessment Agent
- Reviewer Agent

The notebook uses `PlanBasedOrchestrator` to coordinate planning and specialist execution.

## Setup

In [10]:
import os
from pathlib import Path
from dotenv import load_dotenv

from pydantic import BaseModel
from typing import Optional, Literal, List, Dict, Any

from picoagents import Agent, OpenAIChatCompletionClient

# Load .env from the repository root or parent directory.
# Adjust this path if your notebook is located elsewhere.
load_dotenv(Path.cwd() / ".." / ".env")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

if OPENAI_API_KEY:
    print("API key loaded successfully.")
else:
    print("OPENAI_API_KEY is empty. Deterministic workflow examples can still run, but LLM-agent examples need an API key.")

client = OpenAIChatCompletionClient(
    model="gpt-5-mini",
    api_key=OPENAI_API_KEY
)


from picoagents.orchestration import PlanBasedOrchestrator
from picoagents.termination import MaxMessageTermination, TextMentionTermination

API key loaded successfully.


## Define task model and helper functions

In [11]:
class MiniLessonTask(BaseModel):
    """Input model for mini-lesson generation requests."""
    topic: str


def build_plan_based_orchestrator(
    agents: List[Agent],
    model_client: OpenAIChatCompletionClient,
    max_messages: int = 12,
    max_step_retries: int = 2,
) -> PlanBasedOrchestrator:
    """Create a plan-based orchestrator with explicit lesson stop rules."""
    return PlanBasedOrchestrator(
        agents=agents,
        termination=(
            MaxMessageTermination(max_messages) |
            TextMentionTermination("LESSON COMPLETE")
        ),
        model_client=model_client,
        max_step_retries=max_step_retries,
    )

## Create specialist agents

In [12]:
def create_specialist_agents(model_client: OpenAIChatCompletionClient) -> List[Agent]:
    """Create specialist lesson-design agents in planner-friendly order."""
    learning_designer = Agent(
        name="learning_designer",
        instructions="Design clear learning objectives and a short lesson structure.",
        model_client=model_client
    )

    math_explainer = Agent(
        name="math_explainer",
        instructions="Explain the mathematical intuition in a simple way.",
        model_client=model_client
    )

    python_coder = Agent(
        name="python_coder",
        instructions="Create concise Python/scikit-learn examples.",
        model_client=model_client
    )

    assessment_agent = Agent(
        name="assessment_agent",
        instructions="Create quiz questions and short practice tasks.",
        model_client=model_client
    )

    reviewer_agent = Agent(
        name="reviewer_agent",
        instructions=(
            "Review the mini-lesson for clarity and coherence. "
            "When the lesson is complete, end with 'LESSON COMPLETE'."
        ),
        model_client=model_client
    )

    return [
        learning_designer,
        math_explainer,
        python_coder,
        assessment_agent,
        reviewer_agent,
    ]


specialist_agents = create_specialist_agents(client)
learning_designer, math_explainer, python_coder, assessment_agent, reviewer_agent = specialist_agents

## Run plan-based orchestration

### Workflow Diagram

The planner decomposes the lesson task, delegates to specialists, then iterates until the lesson is complete.

```mermaid
flowchart
    T[Mini lesson task] --> P[Planner]
    P --> LD[Learning Designer]
    P --> ME[Math Explainer]
    P --> PC[Python Coder]
    P --> AA[Assessment Agent]
    LD --> R[Reviewer]
    ME --> R
    PC --> R
    AA --> R
    R --> P
    P -->|LESSON COMPLETE or max messages| F[Final mini-lesson]
```

How to read it:
- Planner decides who works next and in what sequence.
- Specialists produce partial outputs for their domain.
- Reviewer checks coherence; planner may trigger another cycle.

In [14]:
lesson_orchestrator = build_plan_based_orchestrator(
    agents=specialist_agents,
    model_client=client,
    max_messages=12,
    max_step_retries=2,
 )

lesson_task = MiniLessonTask(
    topic=(
        "Create a mini-lesson for master's students on logistic regression. "
        "Include intuition, simple math, one scikit-learn example, and two quiz questions."
    )
)

async for message in lesson_orchestrator.run_stream(lesson_task.topic):
    print(message)

[user] 21:26:20 | Create a mini-lesson for master's students on logistic regression. Include intuition, simple math, one scikit-learn example, and two quiz questions.
[learning_designer] 21:27:23 | Learning objectives (by the end of this mini-lesson)
- Explain the intuition behind logistic regression: modelling probabilities for binary outcomes and why we use the sigmoid / log-odds link.
- Derive and write down the core math: logistic (sigmoid) function, logit, likelihood and cross-entropy loss, and brief note on optimization.
- Interpret coefficients as log-odds changes and discuss decision threshold, regularization and multiclass extension at a high level.
- Implement a simple binary classifier with scikit-learn, inspect model attributes (coef_, intercept_, predict_proba) and interpret results.
- Answer short conceptual quiz questions that test intuition and basic math.

One-paragraph intro (to open the lesson)
Logistic regression is the foundational binary classification model that 

## Reflection questions

1. What is the role of the planner?
2. What information should each specialist receive?
3. What can go wrong if the planner creates a poor plan?